# Clase 4 — Práctica: mesetas estériles

En esta práctica vas a comprobar en primera persona cómo el operador de lectura que eliges afecta a la gravedad de la meseta estéril.

## Enunciado

En el experimento de la clase, el operador de lectura es la suma de un operador Z por cada qubit: `sum(cirq.Z(q) for q in qubits)`. Tu tarea es cambiarlo por un único operador que combine dos qubits: `cirq.Z(qubits[0]) * cirq.Z(qubits[1])`. Después, vuelve a ejecutar el barrido de tamaños de qubits y compara la nueva curva de varianza con la original.

## Pistas

- El operador de lectura se pasa como una lista al parámetro `operators` de `tfq.layers.SampledExpectation`; con la suma de Z necesitas envolverla entre corchetes: `[sum(cirq.Z(q) for q in qubits)]`
- El operador Z⊗Z ya viene envuelto de forma natural en una lista de un elemento: `[cirq.Z(qubits[0]) * cirq.Z(qubits[1])]`
- Fíjate en si la nueva curva cae más rápido, más lento o igual que la original, y relaciona lo que observas con la recomendación de "elegir con cuidado qué mides" de la teoría

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt

def generar_circuito_aleatorio(qubits, profundidad, parametro):
    """Un circuito con una puerta aleatoria distinta en cada qubit y en cada capa,
    entrelazado con puertas CZ. Solo el primer parámetro es simbólico (entrenable);
    el resto son ángulos aleatorios fijos."""
    circuito = cirq.Circuit()
    for qubit in qubits:
        circuito += cirq.ry(np.pi / 4.0)(qubit)

    for capa in range(profundidad):
        for i, qubit in enumerate(qubits):
            es_el_parametro_entrenable = (i == 0 and capa == 0)
            angulo = parametro if es_el_parametro_entrenable else np.random.uniform(0, 2 * np.pi)
            eleccion = np.random.uniform()
            if eleccion > 2 / 3:
                circuito += cirq.rz(angulo)(qubit)
            elif eleccion > 1 / 3:
                circuito += cirq.ry(angulo)(qubit)
            else:
                circuito += cirq.rx(angulo)(qubit)
        for origen, destino in zip(qubits, qubits[1:]):
            circuito += cirq.CZ(origen, destino)
    return circuito

def varianza_del_gradiente(num_qubits, profundidad, num_circuitos, repeticiones, operador_lectura):
    """Igual que en la clase, pero ahora recibe una función `operador_lectura(qubits)`
    que construye el operador de lectura, para poder comparar distintas elecciones."""
    qubits = [cirq.GridQubit(0, i) for i in range(num_qubits)]
    simbolo = sympy.symbols("parametro")
    circuitos = [generar_circuito_aleatorio(qubits, profundidad, simbolo) for _ in range(num_circuitos)]

    operadores = operador_lectura(qubits)
    diferenciador = tfq.differentiators.ParameterShift()
    capa_expectativa = tfq.layers.SampledExpectation(differentiator=diferenciador)

    circuitos_tensor = tfq.convert_to_tensor(circuitos)
    valores = tf.convert_to_tensor(np.random.uniform(0, 2 * np.pi, (num_circuitos, 1)).astype(np.float32))

    with tf.GradientTape() as cinta:
        cinta.watch(valores)
        salida = capa_expectativa(circuitos_tensor, operators=operadores,
                                   repetitions=repeticiones, symbol_names=[simbolo], symbol_values=valores)

    gradientes = cinta.gradient(salida, valores)
    return tf.math.reduce_std(gradientes, axis=0).numpy()[0]

# Parámetros del barrido (igual que en la clase)
num_circuitos = 20
repeticiones = 500
tamanos_qubits = [2, 4, 6, 8]
profundidad_fija = 20

# Operador original de la clase: suma de un Z por cada qubit
def operador_suma_z(qubits):
    return [sum(cirq.Z(q) for q in qubits)]

varianzas_suma_z = [varianza_del_gradiente(n, profundidad_fija, num_circuitos, repeticiones, operador_suma_z) for n in tamanos_qubits]

In [ ]:
# TODO: define una función `operador_zz(qubits)` que devuelva el operador de lectura
# cirq.Z(qubits[0]) * cirq.Z(qubits[1]) envuelto en una lista de un elemento

# TODO: vuelve a recorrer `tamanos_qubits` llamando a varianza_del_gradiente con tu nuevo
# operador y guarda el resultado en una lista `varianzas_zz`

# TODO: dibuja ambas curvas (varianzas_suma_z y varianzas_zz) en la misma gráfica,
# con etiquetas y leyenda, para comparar la caída de varianza

<details>
<summary>💡 Ver solución</summary>

```python
# Nuevo operador de lectura: producto Z0*Z1 en vez de la suma de Z
def operador_zz(qubits):
    return [cirq.Z(qubits[0]) * cirq.Z(qubits[1])]

# Repetimos el barrido de qubits con el nuevo operador
varianzas_zz = [varianza_del_gradiente(n, profundidad_fija, num_circuitos, repeticiones, operador_zz) for n in tamanos_qubits]

# Comparamos ambas curvas
plt.plot(tamanos_qubits, varianzas_suma_z, marker="o", label="sum(Z), un Z por qubit")
plt.plot(tamanos_qubits, varianzas_zz, marker="o", label="Z0 * Z1")
plt.xlabel("Número de qubits")
plt.ylabel("Varianza del gradiente")
plt.title("Comparación del operador de lectura sobre la meseta estéril")
plt.legend()
plt.show()

# La curva de Z0 * Z1 cae más rápido que la de sum(Z): medir un operador que combina
# varios qubits a la vez sufre una meseta estéril más severa que medir la suma de
# operadores locales, tal y como anticipaba la teoría.
```

</details>